# T08 — Benchmark Keras (TensorFlow GPU) cho BTL03

Notebook dựng lại và chạy đo trực tiếp **7 mô hình Keras (`tf.keras`)** tương ứng với kiến trúc PyTorch trong `BTL03_Keras_to_PyTorch` (`Func_apxm_MLP`, `MNIST_MLP`, `MNIST_CNN`, `MNIST_CNN_LSTM`, `Human_Gender_CNN`, `FER2013_CNN`, `Hand_Depth_SegNet`) trên **Google Colab (GPU T4)** để thu thập các chỉ số thực tế (**Loss**, **Acc / MAE / MSE**, **Số params**, **Thời gian train/epoch**).

## Hướng dẫn chạy nhanh trên Google Colab
1. Chọn **Runtime → Change runtime type → T4 GPU** (trên giao diện tiếng Việt: **Thời gian chạy → Đổi loại thời gian chạy → T4 GPU**).
2. Notebook đã **nhúng sẵn kết quả T4 của 5 mô hình đã đo** (`MNIST_CNN`, `MNIST_CNN_LSTM`, `Human_Gender_CNN`, `FER2013_CNN`, `Hand_Depth_SegNet`) và bật `SKIP_EXISTING = True` trong cell `setup_env` — **không cần tự upload `keras_benchmark.json`**, chỉ cần bấm **Run all** (`Chạy tất cả`) là Colab sẽ **tự bỏ qua 5 mô hình đã có** và chỉ huấn luyện bổ sung `Func_apxm_MLP` + `MNIST_MLP`.
3. Nếu muốn huấn luyện lại toàn bộ 7 mô hình từ đầu, đổi `SKIP_EXISTING = False` ở cell `setup_env` (khi đó cần `data.zip` qua `DRIVE_DATA_LINK`).
4. Sau khi chạy xong, file **`keras_benchmark.json`** (đủ 7 mô hình) sẽ tự tải xuống; chép đè vào `BTL03_Keras_to_PyTorch/results/keras_benchmark/keras_benchmark.json` và chạy:
   ```powershell
   python scripts/build_benchmark_t08.py --update-doc
   ```

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers

# Cố định seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

DEVICE_DESC = "GPU" if gpus else "CPU"
if gpus:
    try:
        details = tf.config.experimental.get_device_details(gpus[0])
        DEVICE_DESC = details.get("device_name", "GPU")
    except Exception:
        DEVICE_DESC = str(gpus[0])

print(f"TensorFlow: {tf.__version__} | Device: {DEVICE_DESC}")

# Dán link Google Drive tới file data.zip bạn đã upload và bật quyền 'Bất kỳ ai có đường liên kết'
DRIVE_DATA_LINK = "https://drive.google.com/file/d/1myzSFkP2r8rETOJ3QanKQexjiZx1nHys/view?usp=sharing"

# (Tùy chọn) Bật True nếu muốn mount trực tiếp Google Drive cá nhân thay vì dùng link chia sẻ
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

BASE_DIR = Path("BTL03_Keras_to_PyTorch") if (not Path("src").exists() and Path("BTL03_Keras_to_PyTorch").exists()) else Path(".")
DATA_ROOT = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "results/keras_checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_JSON = BASE_DIR / "results/keras_benchmark/keras_benchmark.json"
BENCHMARK_JSON.parent.mkdir(parents=True, exist_ok=True)

# True: tự bỏ qua các mô hình đã có kết quả trong SEED_RESULTS / keras_benchmark.json
# False: chạy huấn luyện lại toàn bộ 7 mô hình từ đầu
SKIP_EXISTING = False

SEED_RESULTS = {
    "MNIST_CNN": {
        "loss": 0.0229, "metric_label": "Acc (%)", "metric": 99.24,
        "params": 1817610, "epochs": 4, "mean_epoch_seconds": 13.92, "device": "Tesla T4",
    },
    "MNIST_CNN_LSTM": {
        "loss": 0.5618, "metric_label": "MAE", "metric": 0.5618,
        "params": 1747073, "epochs": 30, "mean_epoch_seconds": 57.54, "device": "Tesla T4",
    },
    "Human_Gender_CNN": {
        "loss": 0.3355, "metric_label": "Acc (%)", "metric": 87.67,
        "params": 4063570, "epochs": 30, "mean_epoch_seconds": 6.01, "device": "Tesla T4",
    },
    "FER2013_CNN": {
        "loss": 1.1663, "metric_label": "Acc (%)", "metric": 54.74,
        "params": 19028287, "epochs": 30, "mean_epoch_seconds": 20.67, "device": "Tesla T4",
    },
    "Hand_Depth_SegNet": {
        "loss": 0.7282, "metric_label": "Acc (%)", "metric": 80.33,
        "params": 2294279, "epochs": 15, "mean_epoch_seconds": 15.81, "device": "Tesla T4",
    },
}

results_store = dict(SEED_RESULTS) if SKIP_EXISTING else {}
if BENCHMARK_JSON.is_file():
    results_store.update(json.loads(BENCHMARK_JSON.read_text(encoding="utf-8")).get("models", {}))

print(f"BASE_DIR  = {BASE_DIR.resolve()}")
print(f"DATA_ROOT = {DATA_ROOT.resolve()}")
print(f"SKIP_EXISTING = {SKIP_EXISTING} | Đã có sẵn: {list(results_store.keys())}")

TensorFlow: 2.21.0 | Device: Tesla T4
DATA_ROOT = /content/data


In [2]:
import shutil
import subprocess
import sys
import zipfile

DATA_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ("gender", "fer2013", "segnet"):
    (DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)

# Dọn thư mục rỗng 'Untitled Folder' nếu trước đó còn sót
for empty_dir in DATA_ROOT.rglob("Untitled Folder"):
    if empty_dir.is_dir() and not any(empty_dir.iterdir()):
        empty_dir.rmdir()

def _ensure_gdown():
    try:
        import gdown
        return gdown
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "gdown"])
        import gdown
        return gdown

def _unpack_data_zip(zpath):
    if not zipfile.is_zipfile(zpath):
        print(f"[Warn] {zpath} không phải tệp .zip hợp lệ (hãy kiểm tra quyền chia sẻ Drive phải là 'Bất kỳ ai có đường liên kết').")
        return
    tmp_dir = DATA_ROOT / "_unzip_tmp"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)
    print(f"Đang giải nén {zpath} ...")
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(tmp_dir)

    # Tự định vị toàn bộ thư mục dataset ở bất kỳ độ sâu nào trong file zip
    gender_src = next((p.parent for p in tmp_dir.rglob("64_64_11938_4098_train_x_onehot.npy")), None)
    if gender_src is not None:
        shutil.copytree(gender_src, DATA_ROOT / "gender", dirs_exist_ok=True)

    fer_csv_src = next((p.parent for p in tmp_dir.rglob("fer2013_training_onehot.csv")), None)
    fer_dir_src = next((p.parent for p in tmp_dir.rglob("angry") if p.is_dir() and p.parent.name == "train"), None)
    if fer_csv_src is not None:
        shutil.copytree(fer_csv_src, DATA_ROOT / "fer2013", dirs_exist_ok=True)
    elif fer_dir_src is not None:
        shutil.copytree(fer_dir_src.parent, DATA_ROOT / "fer2013", dirs_exist_ok=True)

    seg_src = next((p.parent.parent for p in tmp_dir.rglob("depth") if p.is_dir() and p.parent.name == "train"), None)
    if seg_src is not None:
        shutil.copytree(seg_src, DATA_ROOT / "segnet", dirs_exist_ok=True)

    mnist_raw = next((p.parent for p in tmp_dir.rglob("train-images-idx3-ubyte") if p.is_file()), None)
    if mnist_raw is not None:
        shutil.copytree(mnist_raw, DATA_ROOT / "mnist" / "raw", dirs_exist_ok=True)

    shutil.rmtree(tmp_dir, ignore_errors=True)

need_drive_data = (not SKIP_EXISTING) or any(
    m not in results_store for m in ("Human_Gender_CNN", "FER2013_CNN", "Hand_Depth_SegNet")
)

if not need_drive_data:
    print("[Skip data.zip] Các mô hình dùng data.zip đều đã có kết quả (SKIP_EXISTING=True).")
else:
    # 1. Tải duy nhất file data.zip từ DRIVE_DATA_LINK
    if DRIVE_DATA_LINK.strip():
        gdown = _ensure_gdown()
        link = DRIVE_DATA_LINK.strip()
        dl_zip = DATA_ROOT / "data_drive.zip"
        try:
            if "/folders/" in link:
                dl_dir = DATA_ROOT / "_drive_folder"
                dl_dir.mkdir(parents=True, exist_ok=True)
                gdown.download_folder(url=link, output=str(dl_dir), quiet=False, remaining_ok=True)
                for zp in sorted(dl_dir.rglob("*.zip")):
                    _unpack_data_zip(zp)
                shutil.rmtree(dl_dir, ignore_errors=True)
            else:
                downloaded = gdown.download(url=link, output=str(dl_zip), quiet=False, fuzzy=True)
                if downloaded and Path(downloaded).is_file():
                    _unpack_data_zip(Path(downloaded))
                    Path(downloaded).unlink(missing_ok=True)
        except Exception as exc:
            print("[Warn Drive] Lỗi tải từ DRIVE_DATA_LINK:", exc)

    # 2. Hoặc tự giải nén nếu người dùng đã đặt sẵn data.zip trong /content/ hoặc trên Drive đã mount
    for zcandidate in sorted(set(list(Path(".").glob("*.zip")) + list(DATA_ROOT.glob("*.zip")))):
        _unpack_data_zip(zcandidate)

# 3. Kiểm tra trạng thái sẵn sàng của 7 mô hình
gender_ok = bool(list((DATA_ROOT / "gender").rglob("64_64_11938_4098_train_x_onehot.npy")))
fer_csv_ok = bool(list((DATA_ROOT / "fer2013").rglob("fer2013_training_onehot.csv")))
fer_dir_ok = bool(list((DATA_ROOT / "fer2013").rglob("train"))) and bool(list((DATA_ROOT / "fer2013").rglob("test")))
seg_ok = (DATA_ROOT / "segnet" / "train" / "depth").is_dir() and (DATA_ROOT / "segnet" / "test" / "depth").is_dir()

print("\n=== TRẠNG THÁI DỮ LIỆU ===")
print("[OK]   Func_apxm_MLP (dữ liệu tổng hợp Gaussian 2D)")
print("[OK]   MNIST_MLP, MNIST_CNN & MNIST_CNN_LSTM (tải qua Keras / data.zip)")
print(f"[{'OK' if gender_ok else ('SKIP' if SKIP_EXISTING and 'Human_Gender_CNN' in results_store else 'THIẾU')}] Human_Gender_CNN")
print(f"[{'OK' if (fer_csv_ok or fer_dir_ok) else ('SKIP' if SKIP_EXISTING and 'FER2013_CNN' in results_store else 'THIẾU')}] FER2013_CNN")
print(f"[{'OK' if seg_ok else ('SKIP' if SKIP_EXISTING and 'Hand_Depth_SegNet' in results_store else 'THIẾU')}] Hand_Depth_SegNet")
if need_drive_data and not (gender_ok and (fer_csv_ok or fer_dir_ok) and seg_ok):
    print("-> Hãy dán link Google Drive của tệp data.zip vào DRIVE_DATA_LINK ở cell đầu rồi chạy lại từ đầu!")

Downloading...
From (original): https://drive.google.com/uc?id=1myzSFkP2r8rETOJ3QanKQexjiZx1nHys
From (redirected): https://drive.google.com/uc?id=1myzSFkP2r8rETOJ3QanKQexjiZx1nHys&confirm=t&uuid=012d11ae-40a8-4653-875b-159bd0a9f36b
To: /content/data/data_drive.zip
100%|██████████| 171M/171M [00:02<00:00, 59.1MB/s]


Đang giải nén data/data_drive.zip ...

=== TRẠNG THÁI DỮ LIỆU ===
[OK]   MNIST_CNN & MNIST_CNN_LSTM (tải qua Keras / data.zip)
[OK] Human_Gender_CNN
[OK] FER2013_CNN
[OK] Hand_Depth_SegNet


In [3]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.perf_counter() - self._t0)


def stratified_split_indices(targets, val_ratio=0.1, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    targets = np.asarray(targets)
    for cls in np.unique(targets):
        idx = rng.permutation(np.flatnonzero(targets == cls)).tolist()
        val_count = max(1, round(len(idx) * val_ratio))
        val_idx.extend(idx[:val_count])
        train_idx.extend(idx[val_count:])
    return np.array(train_idx, dtype=np.int64), np.array(val_idx, dtype=np.int64)


def load_mnist_arrays():
    raw_tr_x = next(DATA_ROOT.rglob("train-images-idx3-ubyte"), None)
    if raw_tr_x is not None:
        raw_dir = raw_tr_x.parent
        x_tr = np.fromfile(raw_dir / "train-images-idx3-ubyte", dtype=np.uint8, offset=16).reshape(-1, 28, 28)
        y_tr = np.fromfile(raw_dir / "train-labels-idx1-ubyte", dtype=np.uint8, offset=8)
        x_te = np.fromfile(raw_dir / "t10k-images-idx3-ubyte", dtype=np.uint8, offset=16).reshape(-1, 28, 28)
        y_te = np.fromfile(raw_dir / "t10k-labels-idx1-ubyte", dtype=np.uint8, offset=8)
        return (x_tr, y_tr), (x_te, y_te)
    return keras.datasets.mnist.load_data()


def should_skip(model_name):
    if SKIP_EXISTING and model_name in results_store:
        print(f"[Skip {model_name}] Đã có kết quả:", json.dumps(results_store[model_name], ensure_ascii=False))
        return True
    return False


def save_benchmark_result(model_name, payload):
    results_store[model_name] = payload
    out = {"device": DEVICE_DESC, "tensorflow_version": tf.__version__, "models": results_store}
    BENCHMARK_JSON.write_text(json.dumps(out, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"[Saved {model_name}] ->", json.dumps(payload, ensure_ascii=False))

In [ ]:
# 1. Func_apxm_MLP (65 params với hidden=16, SGD lr=0.01, MSE, bs=32, 50 epochs — khớp EXP_01 PyTorch)
def run_func_apxm_mlp(hidden=16, epochs=50, batch_size=32):
    if should_skip("Func_apxm_MLP"):
        return

    rng = np.random.default_rng(SEED)
    def _make_data(n):
        x = rng.uniform(-1.0, 1.0, (n, 2)).astype(np.float32)
        y = (np.exp(-np.square(x).sum(axis=1) / 2.0) / (2.0 * np.pi)).astype(np.float32)[:, None]
        return x, y

    x_train, y_train = _make_data(1024)
    x_test, y_test = _make_data(100)

    inp = layers.Input(shape=(2,))
    x = layers.Dense(hidden, activation="sigmoid")(inp)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out, name="Func_apxm_MLP")

    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="mse")

    timer = EpochTimer()
    model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, callbacks=[timer], verbose=0)
    test_mse = model.evaluate(x_test, y_test, batch_size=100, verbose=0)

    save_benchmark_result("Func_apxm_MLP", {
        "loss": round(float(test_mse), 4),
        "metric_label": "MSE",
        "metric": round(float(test_mse), 4),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "batch_size": batch_size,
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 4),
        "device": DEVICE_DESC,
    })

run_func_apxm_mlp(hidden=16, epochs=50, batch_size=32)

[Saved Func_apxm_MLP] -> {"loss": 0.0009, "metric_label": "MSE", "metric": 0.0009, "params": 65, "epochs": 50, "batch_size": 32, "mean_epoch_seconds": 0.1347, "device": "Tesla T4"}


In [ ]:
# 2. MNIST_MLP (669,706 params, SGD lr=0.01, bs=128, 30 epochs — khớp T03_mnist PyTorch)
def run_mnist_mlp(epochs=30, batch_size=128):
    if should_skip("MNIST_MLP"):
        return

    (x_train, y_train), (x_test, y_test) = load_mnist_arrays()
    x_train = x_train.astype(np.float32) / 255.0
    x_test = x_test.astype(np.float32) / 255.0

    inp = layers.Input(shape=(28, 28))
    x = layers.Flatten()(inp)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inp, out, name="MNIST_MLP")

    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=0.01),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    timer = EpochTimer()
    model.fit(x_train, y_train, validation_data=(x_test, y_test), batch_size=batch_size, epochs=epochs, callbacks=[timer], verbose=1)
    test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)

    save_benchmark_result("MNIST_MLP", {
        "loss": round(float(test_loss), 4),
        "metric_label": "Acc (%)",
        "metric": round(float(test_acc) * 100.0, 2),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "batch_size": batch_size,
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 2),
        "device": DEVICE_DESC,
    })

run_mnist_mlp(epochs=30, batch_size=128)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.6867 - loss: 1.2074 - val_accuracy: 0.8658 - val_loss: 0.5456
Epoch 2/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8514 - loss: 0.5323 - val_accuracy: 0.8962 - val_loss: 0.3772
Epoch 3/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8783 - loss: 0.4251 - val_accuracy: 0.9088 - val_loss: 0.3232
Epoch 4/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8938 - loss: 0.3733 - val_accuracy: 0.9170 - val_loss: 0.2917
Epoch 5/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9013 - loss: 0.3425 - val_accuracy: 0.9225 - val_loss: 0.2700
Epoch 6/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9075 - loss: 0.3181 - val_accuracy: 0.9281 - val_loss: 0.2522
Epoch 7/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9147 - loss: 0.2958 - val_accuracy: 0.9326 - val_loss: 0.2379
Epoch 8/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step 

In [4]:
# 3. MNIST_CNN (1,817,610 params, Adam lr=0.001, 4 epochs — khớp cấu hình PyTorch)
def run_mnist_cnn(epochs=4, batch_size=128):
    if should_skip("MNIST_CNN"):
        return
    (x_all, y_all), (x_test, y_test) = load_mnist_arrays()
    x_all = (x_all.astype(np.float32) / 255.0)[..., None]
    x_test = (x_test.astype(np.float32) / 255.0)[..., None]

    tr_idx, va_idx = stratified_split_indices(y_all, val_ratio=0.1, seed=SEED)
    x_train, y_train = x_all[tr_idx], y_all[tr_idx]
    x_val, y_val = x_all[va_idx], y_all[va_idx]

    inp = layers.Input(shape=(28, 28, 1))
    x = inp
    for filters in (64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inp, out, name="MNIST_CNN")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt = str(OUT_DIR / "mnist_cnn_best.weights.h5")
    timer = EpochTimer()
    cbs = [
        timer,
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True),
    ]
    model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=batch_size, epochs=epochs, callbacks=cbs, verbose=1)
    model.load_weights(ckpt)
    test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)

    save_benchmark_result("MNIST_CNN", {
        "loss": round(float(test_loss), 4),
        "metric_label": "Acc (%)",
        "metric": round(float(test_acc) * 100.0, 2),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 2),
        "device": DEVICE_DESC,
    })

run_mnist_cnn(epochs=4)

Epoch 1/4
422/422 ━━━━━━━━━━━━━━━━━━━━ 40s 61ms/step - accuracy: 0.9258 - loss: 0.2249 - val_accuracy: 0.9813 - val_loss: 0.0600
Epoch 2/4
422/422 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9825 - loss: 0.0564 - val_accuracy: 0.9858 - val_loss: 0.0470
Epoch 3/4
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9873 - loss: 0.0413 - val_accuracy: 0.9873 - val_loss: 0.0468
Epoch 4/4
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9898 - loss: 0.0325 - val_accuracy: 0.9897 - val_loss: 0.0406
[Saved MNIST_CNN] -> {"loss": 0.0229, "metric_label": "Acc (%)", "metric": 99.24, "params": 1817610, "epochs": 4, "mean_epoch_seconds": 13.92, "device": "Tesla T4"}


In [5]:
# 4. MNIST_CNN_LSTM (1,747,073 params, SGD lr=0.01, MAE loss, 30 epochs)
def _sample_sequences(images, targets, max_to_add=8, seed=2018):
    rng = np.random.RandomState(seed)
    n = len(images)
    seq_x = np.zeros((n, max_to_add, 28, 28, 1), dtype=np.float32)
    seq_y = np.empty(n, dtype=np.float32)
    for i in range(n):
        num_to_add = int(np.ceil(rng.rand() * max_to_add))
        picked = rng.choice(n, size=num_to_add, replace=True)
        seq_x[i, :num_to_add] = images[picked]
        seq_y[i] = float(targets[picked].sum())
    return seq_x, seq_y


def run_mnist_cnn_lstm(epochs=30, batch_size=128, max_to_add=8):
    if should_skip("MNIST_CNN_LSTM"):
        return
    (x_all, y_all), (x_test_raw, y_test_raw) = load_mnist_arrays()
    x_all = (x_all.astype(np.float32) / 255.0)[..., None]
    x_test_raw = (x_test_raw.astype(np.float32) / 255.0)[..., None]

    tr_idx, va_idx = stratified_split_indices(y_all, val_ratio=0.1, seed=SEED)
    x_train, y_train = _sample_sequences(x_all[tr_idx], y_all[tr_idx], max_to_add=max_to_add, seed=2018)
    x_val, y_val = _sample_sequences(x_all[va_idx], y_all[va_idx], max_to_add=max_to_add, seed=2019)
    x_test, y_test = _sample_sequences(x_test_raw, y_test_raw, max_to_add=max_to_add, seed=2020)

    inp = layers.Input(shape=(max_to_add, 28, 28, 1))
    x = inp
    for filters in (64, 128, 256):
        x = layers.TimeDistributed(layers.Conv2D(filters, 3, padding="same", activation="relu"))(x)
        x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
        x = layers.TimeDistributed(layers.Dropout(0.2))(x)
    x = layers.TimeDistributed(layers.Flatten())(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(128, return_sequences=False)(x)
    x = layers.Dropout(0.22)(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out, name="MNIST_CNN_LSTM")

    model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="mae")

    ckpt = str(OUT_DIR / "mnist_lstm_best.weights.h5")
    timer = EpochTimer()
    cbs = [
        timer,
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True),
    ]
    model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=batch_size, epochs=epochs, callbacks=cbs, verbose=1)
    model.load_weights(ckpt)
    test_mae = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)

    save_benchmark_result("MNIST_CNN_LSTM", {
        "loss": round(float(test_mae), 4),
        "metric_label": "MAE",
        "metric": round(float(test_mae), 4),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 2),
        "device": DEVICE_DESC,
    })

run_mnist_cnn_lstm(epochs=30)

Epoch 1/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 63s 136ms/step - loss: 7.0270 - val_loss: 4.1535
Epoch 2/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 3.8492 - val_loss: 3.7005
Epoch 3/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 3.0632 - val_loss: 2.8902
Epoch 4/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 2.5798 - val_loss: 2.4358
Epoch 5/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 2.1699 - val_loss: 1.5214
Epoch 6/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 82s 132ms/step - loss: 1.9093 - val_loss: 1.8181
Epoch 7/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 55s 131ms/step - loss: 1.7360 - val_loss: 1.3064
Epoch 8/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 1.6014 - val_loss: 1.6158
Epoch 9/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 1.5318 - val_loss: 1.9393
Epoch 10/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 1.4462 - val_loss: 0.9887
Epoch 11/30
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 132ms/step - loss: 1.3972 - val_loss: 0.9355
Epoch 12/30
422/422

In [6]:
# 5. Human_Gender_CNN (4,063,570 params, Adam lr=0.001 + EarlyStopping + ReduceLROnPlateau, 30 epochs)
def run_gender_cnn(epochs=30, batch_size=128):
    if should_skip("Human_Gender_CNN"):
        return
    names = [
        "64_64_11938_4098_train_x_onehot.npy",
        "64_64_11938_4098_train_y_onehot.npy",
        "64_64_5968_4098_val_x_onehot.npy",
        "64_64_5968_4098_val_y_onehot.npy",
    ]
    files = [next(DATA_ROOT.rglob(n), DATA_ROOT / "gender" / n) for n in names]
    if not all(f.is_file() for f in files):
        print("[Skip Human_Gender_CNN] Không tìm thấy đủ 4 file NPY trong", DATA_ROOT)
        return

    x_train = np.rot90(np.load(files[0]).astype(np.float32), k=3, axes=(1, 2))
    y_train = np.argmax(np.load(files[1]), axis=1)
    x_val = np.rot90(np.load(files[2]).astype(np.float32), k=3, axes=(1, 2))
    y_val = np.argmax(np.load(files[3]), axis=1)

    inp = layers.Input(shape=(64, 64, 1))
    x = inp
    for filters, reps in ((16, 2), (32, 3), (64, 3), (128, 3), (256, 3)):
        for _ in range(reps):
            x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(1024, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(2, activation="softmax")(x)
    model = keras.Model(inp, out, name="Human_Gender_CNN")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt = str(OUT_DIR / "gender_cnn_best.weights.h5")
    timer = EpochTimer()
    cbs = [
        timer,
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=5, min_lr=1e-4),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
    ]
    model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=batch_size, epochs=epochs, callbacks=cbs, verbose=1)
    model.load_weights(ckpt)
    val_loss, val_acc = model.evaluate(x_val, y_val, batch_size=batch_size, verbose=0)

    save_benchmark_result("Human_Gender_CNN", {
        "loss": round(float(val_loss), 4),
        "metric_label": "Acc (%)",
        "metric": round(float(val_acc) * 100.0, 2),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 2),
        "device": DEVICE_DESC,
    })

run_gender_cnn(epochs=30)

Epoch 1/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 68s 422ms/step - accuracy: 0.7831 - loss: 0.5608 - val_accuracy: 0.7885 - val_loss: 0.5097 - learning_rate: 0.0010
Epoch 2/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.7885 - loss: 0.5132 - val_accuracy: 0.7885 - val_loss: 0.5101 - learning_rate: 0.0010
Epoch 3/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.7885 - loss: 0.5118 - val_accuracy: 0.7885 - val_loss: 0.5097 - learning_rate: 0.0010
Epoch 4/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.7885 - loss: 0.5123 - val_accuracy: 0.7885 - val_loss: 0.5086 - learning_rate: 0.0010
Epoch 5/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.7885 - loss: 0.5142 - val_accuracy: 0.7885 - val_loss: 0.5112 - learning_rate: 0.0010
Epoch 6/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.7885 - loss: 0.5122 - val_accuracy: 0.7885 - val_loss: 0.5081 - learning_rate: 0.0010
Epoch 7/30
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.7885 - loss: 0.5077 - val_a

In [7]:
# 6. FER2013_CNN (19,028,287 params, Adam lr=0.0001 + Augmentation + EarlyStopping)
def _load_fer2013_arrays(fdir):
    csv_tr = next(fdir.rglob("fer2013_training_onehot.csv"), fdir / "fer2013_training_onehot.csv")
    csv_te = next(fdir.rglob("fer2013_publictest_onehot.csv"), fdir / "fer2013_publictest_onehot.csv")
    if csv_tr.is_file() and csv_te.is_file():
        tr = np.loadtxt(csv_tr, delimiter=",", dtype=np.float32)
        te = np.loadtxt(csv_te, delimiter=",", dtype=np.float32)
        x_all = tr[:, :2304].reshape(-1, 48, 48, 1)
        y_all = tr[:, 2304:].argmax(axis=1)
        x_test = te[:, :2304].reshape(-1, 48, 48, 1)
        y_test = te[:, 2304:].argmax(axis=1)
        return x_all, y_all, x_test, y_test

    tr_dir = next((p for p in fdir.rglob("train") if p.is_dir()), fdir / "train")
    te_dir = next((p for p in fdir.rglob("test") if p.is_dir()), fdir / "test")
    if not (tr_dir.is_dir() and te_dir.is_dir()):
        return None
    classes = sorted([p.name for p in tr_dir.iterdir() if p.is_dir()])
    def _read_folder(folder):
        xs, ys = [], []
        for idx, cls in enumerate(classes):
            for img_path in sorted((folder / cls).glob("*")):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    img = Image.open(img_path).convert("L").resize((48, 48), Image.BILINEAR)
                    xs.append(np.asarray(img, dtype=np.float32) / 255.0)
                    ys.append(idx)
        return np.asarray(xs)[..., None], np.asarray(ys, dtype=np.int64)
    return (*_read_folder(tr_dir), *_read_folder(te_dir))


def run_fer2013_cnn(epochs=30, batch_size=128):
    if should_skip("FER2013_CNN"):
        return
    fdir = DATA_ROOT / "fer2013"
    loaded = _load_fer2013_arrays(fdir)
    if loaded is None:
        print("[Skip FER2013_CNN] Không tìm thấy CSV hoặc thư mục train/test trong", fdir)
        return
    x_all, y_all, x_test, y_test = loaded
    tr_idx, va_idx = stratified_split_indices(y_all, val_ratio=0.1, seed=SEED)
    x_train, y_train = x_all[tr_idx], y_all[tr_idx]
    x_val, y_val = x_all[va_idx], y_all[va_idx]

    inp = layers.Input(shape=(48, 48, 1))
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(inp)
    x = layers.Conv2D(64, 3, padding="valid", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    for filters, pad in ((128, "same"), (128, "valid"), (218, "same"), (218, "valid")):
        x = layers.Conv2D(filters, 3, padding=pad, activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(7, activation="softmax")(x)
    model = keras.Model(inp, out, name="FER2013_CNN")

    aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(45.0 / 360.0),
        layers.RandomTranslation(0.2, 0.2),
    ])
    train_ds = (
        tf.data.Dataset.from_tensor_slices((x_train, y_train))
        .shuffle(len(x_train), seed=SEED)
        .batch(batch_size)
        .map(lambda bx, by: (aug(bx, training=True), by), num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt = str(OUT_DIR / "fer2013_cnn_best.weights.h5")
    timer = EpochTimer()
    cbs = [
        timer,
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", save_best_only=True, save_weights_only=True),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
    ]
    model.fit(train_ds, validation_data=(x_val, y_val), epochs=epochs, callbacks=cbs, verbose=1)
    model.load_weights(ckpt)
    test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)

    save_benchmark_result("FER2013_CNN", {
        "loss": round(float(test_loss), 4),
        "metric_label": "Acc (%)",
        "metric": round(float(test_acc) * 100.0, 2),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 2),
        "device": DEVICE_DESC,
    })

run_fer2013_cnn(epochs=30)

Epoch 1/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 67s 227ms/step - accuracy: 0.2488 - loss: 1.8188 - val_accuracy: 0.2597 - val_loss: 1.7853
Epoch 2/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 36s 92ms/step - accuracy: 0.2514 - loss: 1.7923 - val_accuracy: 0.2639 - val_loss: 1.7721
Epoch 3/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 19s 94ms/step - accuracy: 0.2538 - loss: 1.7824 - val_accuracy: 0.2820 - val_loss: 1.7539
Epoch 4/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 20s 93ms/step - accuracy: 0.2621 - loss: 1.7701 - val_accuracy: 0.3040 - val_loss: 1.7166
Epoch 5/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 19s 92ms/step - accuracy: 0.2710 - loss: 1.7565 - val_accuracy: 0.3071 - val_loss: 1.6994
Epoch 6/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 19s 92ms/step - accuracy: 0.2829 - loss: 1.7419 - val_accuracy: 0.3210 - val_loss: 1.6882
Epoch 7/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 19s 92ms/step - accuracy: 0.2958 - loss: 1.7233 - val_accuracy: 0.3339 - val_loss: 1.6422
Epoch 8/30
202/202 ━━━━━━━━━━━━━━━━━━━━ 19s 93ms/step - accuracy: 0.3031 - loss: 1.7103 -

In [8]:
# 7. Hand_Depth_SegNet (Kiến trúc 3 tầng khớp notebooks/04_segmentation.ipynb, Adam lr=1e-4, batch_size=8, 15 epochs)
def _load_segnet_split(sdir, target_size=(96, 96)):
    depth_files = sorted((sdir / "depth").glob("*.png"))
    if not depth_files:
        return None
    raw_values = [0, 43, 85, 128, 170, 212, 255]
    lut = np.zeros(256, dtype=np.int64)
    for cls_idx, val in enumerate(raw_values):
        lut[val] = cls_idx

    xs = np.empty((len(depth_files), target_size[1], target_size[0], 1), dtype=np.float32)
    ys = np.empty((len(depth_files), target_size[1], target_size[0]), dtype=np.int64)
    for i, dp in enumerate(depth_files):
        lp = sdir / "labels" / dp.name.replace("_depth_", "_labels_")
        d_img = Image.open(dp).convert("L").resize(target_size, Image.BILINEAR)
        l_img = Image.open(lp).resize(target_size, Image.NEAREST)
        l_arr = np.asarray(l_img)
        if l_arr.ndim == 3:
            l_arr = l_arr[:, :, 0]
        xs[i, :, :, 0] = np.asarray(d_img, dtype=np.float32) / 255.0
        ys[i] = lut[l_arr]
    return xs, ys


def run_hand_depth_segnet(epochs=15, batch_size=8):
    if should_skip("Hand_Depth_SegNet"):
        return
    sdir = next((p.parent.parent for p in DATA_ROOT.rglob("depth") if p.is_dir() and p.parent.name == "train"), DATA_ROOT / "segnet")
    tr = _load_segnet_split(sdir / "train")
    te = _load_segnet_split(sdir / "test")
    if tr is None or te is None:
        print("[Skip Hand_Depth_SegNet] Không tìm thấy ảnh trong", sdir)
        return
    x_train, y_train = tr
    x_test, y_test = te

    inp = layers.Input(shape=(96, 96, 1))
    x = inp
    for filters in (64, 128, 256):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same")(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)

    for f_in, f_out in ((256, 128), (128, 64)):
        x = layers.UpSampling2D(2)(x)
        for filters in (f_in, f_out):
            x = layers.Conv2D(filters, 3, padding="same")(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    out = layers.Conv2D(7, 1, activation="softmax")(x)
    model = keras.Model(inp, out, name="Hand_Depth_SegNet")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    timer = EpochTimer()
    model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, callbacks=[timer], verbose=1)
    test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=0)

    save_benchmark_result("Hand_Depth_SegNet", {
        "loss": round(float(test_loss), 4),
        "metric_label": "Acc (%)",
        "metric": round(float(test_acc) * 100.0, 2),
        "params": int(model.count_params()),
        "epochs": len(timer.epoch_times),
        "mean_epoch_seconds": round(float(np.mean(timer.epoch_times)), 2),
        "device": DEVICE_DESC,
    })

run_hand_depth_segnet(epochs=15)

Epoch 1/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 62s 98ms/step - accuracy: 0.7585 - loss: 0.7324
Epoch 2/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 37s 35ms/step - accuracy: 0.8229 - loss: 0.5192
Epoch 3/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.8524 - loss: 0.4304
Epoch 4/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.8743 - loss: 0.3665
Epoch 5/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.8920 - loss: 0.3160
Epoch 6/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9056 - loss: 0.2758
Epoch 7/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9150 - loss: 0.2472
Epoch 8/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9216 - loss: 0.2261
Epoch 9/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9268 - loss: 0.2093
Epoch 10/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - accuracy: 0.9321 - loss: 0.1930
Epoch 11/15
307/307 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.9364 - loss: 0.1792
Epoch 12/15
307/307 ━━━━━━━━━━

In [9]:
print(BENCHMARK_JSON.read_text(encoding="utf-8"))
try:
    from google.colab import files
    files.download(str(BENCHMARK_JSON))
except ImportError:
    print("Hoàn tất. Lưu file tại:", BENCHMARK_JSON.resolve())

{
  "device": "Tesla T4",
  "tensorflow_version": "2.21.0",
  "models": {
    "MNIST_CNN": {
      "loss": 0.0229,
      "metric_label": "Acc (%)",
      "metric": 99.24,
      "params": 1817610,
      "epochs": 4,
      "mean_epoch_seconds": 13.92,
      "device": "Tesla T4"
    },
    "MNIST_CNN_LSTM": {
      "loss": 0.5618,
      "metric_label": "MAE",
      "metric": 0.5618,
      "params": 1747073,
      "epochs": 30,
      "mean_epoch_seconds": 57.54,
      "device": "Tesla T4"
    },
    "Human_Gender_CNN": {
      "loss": 0.3355,
      "metric_label": "Acc (%)",
      "metric": 87.67,
      "params": 4063570,
      "epochs": 30,
      "mean_epoch_seconds": 6.01,
      "device": "Tesla T4"
    },
    "FER2013_CNN": {
      "loss": 1.1663,
      "metric_label": "Acc (%)",
      "metric": 54.74,
      "params": 19028287,
      "epochs": 30,
      "mean_epoch_seconds": 20.67,
      "device": "Tesla T4"
    },
    "Hand_Depth_SegNet": {
      "loss": 0.7282,
      "metric_label": "

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>